# Hidden Markov Models (HMMs) and Viterbi Algorithm

## 📚 Learning Objectives

By completing this notebook, you will:
- Understand Hidden Markov Models (HMMs) structure and components
- Implement HMMs for sequence prediction
- Implement Viterbi algorithm for sequence decoding
- Apply HMMs to practical problems (speech recognition, POS tagging)

## 🔗 Prerequisites

- ✅ Basic probability (see `01_learning_under_uncertainty.ipynb`)
- ✅ Markov chains — introduced from scratch in Part 0 below
- ✅ Python 3.8+ installed

---

This notebook covers practical activities from **Course 02, Unit 3**:
- Working with Hidden Markov Models (HMMs) for sequence prediction
- Implementing Viterbi algorithm for sequence decoding
- Applying HMMs to practical problems (speech recognition, POS tagging)

---

## Introduction to Hidden Markov Models

**Hidden Markov Models (HMMs)** are statistical models for sequences where:
- **Hidden states**: Unobserved states (e.g., weather: Sunny, Rainy)
- **Observations**: Observed outputs (e.g., activities: Walk, Shop, Clean)
- **Transitions**: Probabilities between hidden states
- **Emissions**: Probabilities of observations given states

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
import numpy as np

print("✅ Libraries imported!")
print("Ready to work with HMMs and Viterbi algorithm!")


✅ Libraries imported!
Ready to work with HMMs and Viterbi algorithm!


## Part 0: A 60-Second Markov Chain Primer

An HMM is built on a **Markov chain**, so let's meet that idea first.

A Markov chain is a system that hops between **states** (e.g., Sunny / Rainy), where the probability of the next state depends ONLY on the current state — not on the full history. This is the **Markov property** ("memorylessness").

Its entire behavior is captured by a **transition matrix**: one row per current state, giving the probability of each next state (every row sums to 1).

The simulation below shows two things:
1. A sampled sequence of weather states (each day generated only from the previous day).
2. The long-run fraction of days spent in each state settles to fixed values determined by the transition matrix alone.

An HMM then adds one twist: you cannot see the states themselves — only noisy **observations** emitted from them.

In [2]:
# A tiny 2-state Markov chain: tomorrow's weather depends only on today's
np.random.seed(42)  # Reproducible simulation

# Transition matrix: P[current][next]  (each row sums to 1)
P = {
    'Sunny': {'Sunny': 0.7, 'Rainy': 0.3},
    'Rainy': {'Sunny': 0.4, 'Rainy': 0.6}
}

# Simulate 10,000 days, starting Sunny
states_list = ['Sunny', 'Rainy']
day, chain = 'Sunny', ['Sunny']
for _ in range(9999):
    probs = [P[day][s] for s in states_list]
    day = np.random.choice(states_list, p=probs)
    chain.append(day)

print("First 15 simulated days:")
print("  " + " -> ".join(s[0] for s in chain[:15]) + "   (S=Sunny, R=Rainy)")
print()

# Long-run fraction of time in each state (empirical)
frac_sunny = chain.count('Sunny') / len(chain)
frac_rainy = chain.count('Rainy') / len(chain)

# Theoretical long-run (stationary) fractions for a 2-state chain:
# pi_Sunny = P(R->S) / (P(S->R) + P(R->S))
pi_sunny = P['Rainy']['Sunny'] / (P['Sunny']['Rainy'] + P['Rainy']['Sunny'])
pi_rainy = 1 - pi_sunny

print(f"Long-run fraction of days (10,000-day simulation vs. theory):")
print(f"  Sunny: {frac_sunny:.3f}  (theory: {pi_sunny:.3f})")
print(f"  Rainy: {frac_rainy:.3f}  (theory: {pi_rainy:.3f})")
print()
print("Key point: the transition matrix alone controls the chain's behavior.")
print("Next: an HMM uses exactly this kind of chain for its HIDDEN states.")

First 15 simulated days:
  S -> S -> R -> R -> R -> S -> S -> S -> R -> R -> R -> S -> R -> R -> S   (S=Sunny, R=Rainy)

Long-run fraction of days (10,000-day simulation vs. theory):
  Sunny: 0.586  (theory: 0.571)
  Rainy: 0.414  (theory: 0.429)

Key point: the transition matrix alone controls the chain's behavior.
Next: an HMM uses exactly this kind of chain for its HIDDEN states.


## Part 1: Simple HMM Implementation

Let's implement a simple HMM for weather prediction.


In [3]:
# Build a Hidden Markov Model: hidden weather states we cannot see, activities we CAN observe.
# Why: HMMs answer 'what is hidden behind my observations?' - the same math behind speech recognition and tagging.
# The forward algorithm computes P(observation sequence) efficiently with dynamic programming.

class SimpleHMM:
    """Simple Hidden Markov Model implementation"""
    
    def __init__(self, states, observations, transition_probs, emission_probs, initial_probs):
        """
        Parameters:
        - states: List of hidden states
        - observations: List of possible observations
        - transition_probs: Dict of transition probabilities, indexed
          transition_probs[current_state][next_state] = P(next | current)
        - emission_probs: Dict of emission probabilities P(obs | state)
        - initial_probs: Initial state probabilities
        """
        self.states = states
        self.observations = observations
        self.transition_probs = transition_probs
        self.emission_probs = emission_probs
        self.initial_probs = initial_probs
    
    def forward(self, obs_sequence):
        """Forward algorithm: Compute probability of observation sequence"""
        T = len(obs_sequence)
        N = len(self.states)
        
        # Initialize alpha (forward probabilities)
        alpha = np.zeros((T, N))
        
        # Initialization
        for i, state in enumerate(self.states):
            alpha[0, i] = self.initial_probs[state] * self.emission_probs[state][obs_sequence[0]]
        
        # Recursion
        for t in range(1, T):
            for j, state_j in enumerate(self.states):
                alpha[t, j] = sum(
                    alpha[t-1, i] * self.transition_probs[self.states[i]][state_j] 
                    for i in range(N)
                ) * self.emission_probs[state_j][obs_sequence[t]]
        
        # Termination
        return alpha, sum(alpha[T-1, :])

# Example: Weather HMM
states = ['Sunny', 'Rainy']
observations = ['Walk', 'Shop', 'Clean']

# Transition probabilities: P(next_state | current_state)
transition_probs = {
    'Sunny': {'Sunny': 0.7, 'Rainy': 0.3},
    'Rainy': {'Sunny': 0.4, 'Rainy': 0.6}
}

# Emission probabilities: P(observation | state)
emission_probs = {
    'Sunny': {'Walk': 0.6, 'Shop': 0.3, 'Clean': 0.1},
    'Rainy': {'Walk': 0.1, 'Shop': 0.4, 'Clean': 0.5}
}

# Initial probabilities
initial_probs = {'Sunny': 0.6, 'Rainy': 0.4}

# Create HMM
hmm = SimpleHMM(states, observations, transition_probs, emission_probs, initial_probs)

print("=" * 60)
print("Hidden Markov Model: Weather Prediction")
print("=" * 60)
print(f"States: {states}")
print(f"Observations: {observations}")

Hidden Markov Model: Weather Prediction
States: ['Sunny', 'Rainy']
Observations: ['Walk', 'Shop', 'Clean']


## Part 2: The Forward Algorithm — Probability of an Observation Sequence

Before decoding hidden states, the first classic HMM question is: **how likely is an observation sequence under this model?**

The **forward algorithm** answers it with dynamic programming. It fills a table `alpha[t, i]` = probability of seeing the first `t+1` observations AND being in state `i` at time `t`, then sums the last row.

In [4]:
# Run the forward algorithm on an observation sequence
# Why print the alpha table? Each row shows how probability mass flows through the hidden states
# one time step at a time - summing the last row gives the total sequence probability.
obs_seq = ['Walk', 'Shop', 'Clean']

alpha, total_prob = hmm.forward(obs_seq)

print("=" * 60)
print(f"Forward Algorithm: P({obs_seq})")
print("=" * 60)
print()
print(f"{'t':<4}{'observation':<14}" + "".join(f"alpha[{s}]".ljust(16) for s in hmm.states))
for t, obs in enumerate(obs_seq):
    row = "".join(f"{alpha[t, i]:<16.6f}" for i in range(len(hmm.states)))
    print(f"{t:<4}{obs:<14}" + row)
print()
print(f"P(observation sequence) = sum of last row = {total_prob:.6f}")

Forward Algorithm: P(['Walk', 'Shop', 'Clean'])

t   observation   alpha[Sunny]    alpha[Rainy]    
0   Walk          0.360000        0.040000        
1   Shop          0.080400        0.052800        
2   Clean         0.007740        0.027900        

P(observation sequence) = sum of last row = 0.035640


## Part 3: Viterbi Algorithm for Sequence Decoding

The Viterbi algorithm finds the most likely sequence of hidden states given observations.


In [5]:
# Viterbi algorithm: find the single MOST LIKELY hidden state sequence for the observations.
# Why: forward gives 'how likely are these observations overall'; Viterbi answers the decoding
# question 'what was the weather each day?' using max instead of sum, plus backpointers to recover the path.

def viterbi(hmm, obs_sequence):
    """
    Viterbi algorithm: Find most likely sequence of hidden states
    
    Returns:
    - best_path: Most likely state sequence
    - best_prob: Probability of best path
    """
    T = len(obs_sequence)
    N = len(hmm.states)
    
    # Initialize viterbi and backpointer tables
    viterbi_table = np.zeros((T, N))
    backpointer = np.zeros((T, N), dtype=int)
    
    # Initialization
    for i, state in enumerate(hmm.states):
        viterbi_table[0, i] = hmm.initial_probs[state] * hmm.emission_probs[state][obs_sequence[0]]
        backpointer[0, i] = 0
    
    # Recursion
    for t in range(1, T):
        for j, state_j in enumerate(hmm.states):
            # Find best previous state
            probs = [
                viterbi_table[t-1, i] * hmm.transition_probs[hmm.states[i]][state_j]
                for i in range(N)
            ]
            best_prev = np.argmax(probs)
            viterbi_table[t, j] = probs[best_prev] * hmm.emission_probs[state_j][obs_sequence[t]]
            backpointer[t, j] = best_prev
    
    # Termination: Find best final state
    best_final = np.argmax(viterbi_table[T-1, :])
    best_prob = viterbi_table[T-1, best_final]
    
    # Backtrack to find best path
    best_path = [hmm.states[best_final]]
    for t in range(T-1, 0, -1):
        best_final = backpointer[t, best_final]
        best_path.insert(0, hmm.states[best_final])
    
    return best_path, best_prob

# Example: Decode observation sequence
obs_seq = ['Walk', 'Shop', 'Clean']
print("=" * 60)
print(f"Observation Sequence: {obs_seq}")
print("=" * 60)

best_states, prob = viterbi(hmm, obs_seq)
print(f"Most likely state sequence: {best_states}")
print(f"Probability: {prob:.6f}")

Observation Sequence: ['Walk', 'Shop', 'Clean']
Most likely state sequence: ['Sunny', 'Rainy', 'Rainy']
Probability: 0.012960


## Part 4: Application - Part-of-Speech Tagging

Let's apply HMMs to POS tagging (simplified example).


In [6]:
# Example: POS Tagging HMM
pos_states = ['Noun', 'Verb', 'Det']
pos_words = ['the', 'cat', 'runs']

# Transition: P(POS_i | POS_j)
pos_transitions = {
    'Noun': {'Noun': 0.1, 'Verb': 0.3, 'Det': 0.6},
    'Verb': {'Noun': 0.5, 'Verb': 0.2, 'Det': 0.3},
    'Det': {'Noun': 0.7, 'Verb': 0.2, 'Det': 0.1}
}

# Emission: P(word | POS)
pos_emissions = {
    'Noun': {'the': 0.1, 'cat': 0.7, 'runs': 0.2},
    'Verb': {'the': 0.05, 'cat': 0.1, 'runs': 0.85},
    'Det': {'the': 0.8, 'cat': 0.15, 'runs': 0.05}
}

pos_initial = {'Noun': 0.3, 'Verb': 0.3, 'Det': 0.4}

pos_hmm = SimpleHMM(pos_states, pos_words, pos_transitions, pos_emissions, pos_initial)

# Tag sentence: "the cat runs"
sentence = ['the', 'cat', 'runs']
pos_sequence, pos_prob = viterbi(pos_hmm, sentence)

print("=" * 60)
print("POS Tagging Example:")
print("=" * 60)
print(f"Sentence: {sentence}")
print(f"Tagged: {list(zip(sentence, pos_sequence))}")
print(f"Probability: {pos_prob:.6f}")


POS Tagging Example:
Sentence: ['the', 'cat', 'runs']
Tagged: [('the', 'Det'), ('cat', 'Noun'), ('runs', 'Verb')]
Probability: 0.039984


## Summary

### Key Concepts:
1. **HMM Components**: States, observations, transitions, emissions
2. **Forward Algorithm**: Compute probability of observation sequence
3. **Viterbi Algorithm**: Find most likely hidden state sequence
4. **Applications**: Speech recognition, POS tagging, sequence prediction

### Applications:
- Natural language processing (POS tagging, NER)
- Speech recognition
- Bioinformatics (gene prediction)
- Time series prediction

**Reference:** Course 02, Unit 3: "Working with Hidden Markov Models (HMMs)" and "Implementing Viterbi algorithm for sequence decoding"


## 📚 References

1. Viterbi, A. J. (1967). *Error Bounds for Convolutional Codes and an Asymptotically Optimum Decoding Algorithm*. IEEE Transactions on Information Theory 13(2), 260-269. (origin of the Viterbi algorithm)
2. Rabiner, L. R. (1989). *A Tutorial on Hidden Markov Models and Selected Applications in Speech Recognition*. Proceedings of the IEEE 77(2), 257-286. (the classic HMM tutorial)
3. Jurafsky, D., & Martin, J. H. (2024 draft). *Speech and Language Processing* (3rd ed.), Appendix on Hidden Markov Models. <https://web.stanford.edu/~jurafsky/slp3/>